# Prompt Engineering with Groq API

**Prompt engineering** = writing the input (prompt) in a smart way so the LLM gives the output we want. We do not change the model, we only change the input.

LLM is a next-word predictor. A better prompt pushes it to predict better words.

- Bad prompt: `Write about dogs`
- Good prompt: `Write a 100-word paragraph on why dogs suit elderly people, warm tone`

**5 pillars of a good prompt**
1. Clarity - say exactly what you want
2. Context - give the background the model does not know
3. Specificity - fix format, length, tone
4. Iteration - first prompt rarely works, improve it again and again
5. Decomposition - break a big task into small steps

## test cell
Just checking that python / notebook is working.

In [67]:
print("hello world")

hello world


## import groq
Groq gives us a **free hosted LLM through an API**. We do not run the model on our laptop, we just send a request and get the answer back.
`Groq` is the class we use to talk to it.

In [68]:
from groq import Groq

## load api key from .env file
Never write the API key directly in code (anyone can steal it).
Keep it in a `.env` file as `GROQ_API_KEY=xxxx` and load it with `load_dotenv()`.

In [69]:
from dotenv import load_dotenv
load_dotenv()

True

## os module
`os.environ` is used to read the environment variables (our key is stored there).

In [70]:
import os

## check the key
Print once to confirm the key is loaded. Do **not** share this output with anyone.

In [71]:
print(os.environ["GROQ_API_KEY"])

YOUR_GROQ_API_KEY


## create client
`client` is our connection to Groq. Every request we make goes through this object.

In [72]:
client = Groq(api_key=os.environ["GROQ_API_KEY"])

## message format
We never send plain text to the model. We send a **list of messages**, and each message has:
- `role` - who is speaking
- `content` - what they said

| role | meaning |
|------|---------|
| system | rules for the whole chat (how to behave) |
| user | our question |
| assistant | answer given by the model |

In [73]:
message=[
        {
            "role": "user",
            "content": "EXPLAIN ATTENTION IS ALL YOU NEED PAPER",
        }
    ]

## Input prompt

This is the message (prompt) we send to the model. It is a list with one `user` message.
Output of the next cell just shows this list.

In [74]:
message

[{'role': 'user', 'content': 'EXPLAIN ATTENTION IS ALL YOU NEED PAPER'}]

## call the model
`client.chat.completions.create()` sends our messages to the model.
- `messages` - the input we send
- `model` - which LLM to use (here `openai/gpt-oss-20b`)

In [75]:
chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b")

## read the response
The model returns a big object. The actual answer text is at
`choices[0].message.content`.

In [76]:
print(chat_completion.choices[0].message.content)

## “Attention Is All You Need” – A Deep‑Dive

> **Authors**: Ashish Vaswani, Noam Shazeer, Niki Parmar, Jakob Uszkoreit, Llion Jones, Aidan N. Gomez, Łukasz Kaiser, Illia Polosukhin  
> **Venue**: NIPS 2017 (Neural Information Processing Systems)  
> **Why It Matters**: The paper introduced the **Transformer** architecture, a pure‑attention, no‑RNN model that became the foundation for virtually every state‑of‑the‑art NLP system today (BERT, GPT, T5, etc.).

---

### 1. What Problem Does It Solve?

Before Transformers, sequence‑to‑sequence (seq2seq) tasks (e.g. machine translation) relied on **Recurrent Neural Networks (RNNs)** or **Convolutional Neural Networks (CNNs)**. Those architectures had several limitations:

| Limitation | Typical Impact |
|------------|----------------|
| **Sequential computation** | Long training times; hard to parallelize |
| **Difficulty with long‑range dependencies** | Vanishing/exploding gradients; limited context |
| **Complexity of gating mechanisms** |

## function to ask LLM
Writing the same code again and again is boring, so we put it in a function.
Now we only call `ask_llm("question")` and get the answer as text.

In [77]:
def ask_llm(user_query):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b")
    response = chat_completion.choices[0].message.content
    return response

In [78]:
ask_llm("tell m about me")

'I don’t have any personal information about you, so I can’t give you a description of yourself. If you let me know what kind of information you’re looking for—like personality traits, strengths, interests, or something else—I can offer general suggestions or ask you a few questions to help you explore that.'

## chatbot with memory (multi-turn)
The model has **no memory**. It only knows what we send in the current request.
So to make a chatbot, we keep a `history` list and send the **full history every time**.

Steps in the loop:
1. take the user input
2. add it to history as `user`
3. send the whole history to the model
4. add the model reply to history as `assistant`
5. type `exit` to stop

The first message is the **system prompt**: it tells the model its job (here: coding assistant, and say "I do not know" for non-coding questions). It is set once and applies for the whole conversation.

In [79]:
history = []
system_prompt = {
        "role":"system", # user prompt
        "content":"You are AI coding assistant powered by github  . help me solve c , python , java , c++ coding problem. if user ask query unrelated to coding just tell them i do not know"
    }
history.append(system_prompt)
while True:
    user_query = input("Enter user query: ")
    if user_query.lower() == "exit":
        break
    message = {

        "role":"user", # user prompt
        "content":user_query
    }
    history.append(message)
    chat_completion = client.chat.completions.create( messages=history,model="openai/gpt-oss-20b")
    response = chat_completion.choices[0].message.content
    print(response)
    ai_response = {
        "role": "assistant",
        "content":response
    }
    history.append(ai_response)
    

Hello! 👋 How can I help you today?
I do not know.


In [149]:
system prompt ;user prompt ; ai response

[]

In [81]:
history

[{'role': 'system',
  'content': 'You are AI coding assistant powered by github  . help me solve c , python , java , c++ coding problem. if user ask query unrelated to coding just tell them i do not know'},
 {'role': 'user', 'content': 'Hi'},
 {'role': 'assistant', 'content': 'Hello! 👋 How can I help you today?'},
 {'role': 'user', 'content': 'Tell me about gpt'},
 {'role': 'assistant', 'content': 'I do not know.'}]

## good prompt vs bad prompt
Same topic, two prompts. See how the answer changes.
- `prompt1` - short and vague, so the answer is long and random
- `prompt2` - gives **role + task + constraint** (expert, explain python, 5 points)

Rule: the clearer the prompt, the better the output.

In [82]:
prompt1  = """tell me about python?"""

In [83]:
prompt2 = """you are expert in python coding language.help me understand what is python and explain in 5 points."""

In [84]:
print(ask_llm(prompt1))

Sure thing! 👋 Just to make sure I give you the info you’re looking for—when you say “Python,” are you referring to:

1. **Python (the programming language)** – a popular, high‑level language known for its readability and versatility.
2. **Python (the snake)** – a large, non‑venomous constrictor found in many parts of the world.

Let me know which one (or both) you'd like to learn about, and I’ll dive right in!


In [85]:
print(ask_llm(prompt2))

**Python – A Quick Overview (5 Key Points)**

1. **High‑level, interpreted language**  
   - Python code runs in a runtime (the Python interpreter) rather than being compiled to machine code.  
   - This allows rapid development and interactive experimentation.

2. **Readable, concise syntax**  
   - Indentation defines code blocks (no curly braces).  
   - Built‑in data types (lists, dictionaries, sets, tuples) and powerful string handling make common tasks simple.

3. **Extremely versatile ecosystem**  
   - Libraries for data science (`pandas`, `NumPy`, `scikit‑learn`), machine learning (`TensorFlow`, `PyTorch`), web development (`Django`, `Flask`), automation, scripting, networking, etc.  
   - Python’s “batteries‑included” philosophy means many features are available out of the box.

4. **Cross‑platform, open source**  
   - Runs natively on Windows, macOS, Linux, and many other OSes.  
   - The CPython implementation is free and maintained by the Python Software Foundation.

5. *

## temperature and max_tokens
Two settings that control the output:

- **temperature** - randomness of the answer
  - `0` = same, safe, predictable answer every time
  - `1` = more creative, different answer each time
- **max_tokens** - maximum length of the answer. If the limit is too small, the answer gets **cut in the middle**.

(1 token is roughly 3/4 of a word)

In [86]:
def ask_llm_temp_1(user_query):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=1)
    response = chat_completion.choices[0].message.content
    return response

In [87]:
def ask_llm_temp_0(user_query):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=0)
    response = chat_completion.choices[0].message.content
    return response

## compare temperature 1 vs 0
Run the same query on both functions. Temp 1 gives a different story each run, temp 0 gives almost the same story each run.

In [88]:
query  = "Write a story about python in 5 points only"

In [89]:
print(ask_llm_temp_1(query))

**A Quick Tale of Python in Five Stanzas**

1. **Birth of a Language** – In 1991, Guido van Rossum, a Dutch programmer, crafted Python as a successor to ABC, blending simplicity with powerful, readable syntax that invited both beginners and experts alike.  
2. **The Snake’s Rise** – From a quiet niche to a worldwide juggernaut, Python grew as libraries (NumPy, Django, Pandas) sprouted, turning it into the go‑to language for web apps, data science, AI, and automation.  
3. **Community & Culture** – The Python community, driven by the motto “There should be one—and preferably only one—obvious way to do it,” embraced open‑source collaboration, hosting conferences (PyCon) and fostering inclusive mentorship programs.  
4. **Python in the Real World** – Today, major tech giants, finance firms, scientific labs, and governments deploy Python for everything from building autonomous vehicles to analyzing genome sequences, proving its versatility and reliability.  
5. **Future Horizons** – As Pyt

In [90]:
print(ask_llm_temp_0(query))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Simplicity.”* She opens it, and the first line, “Hello, World!” sparks a curiosity that feels like a secret handshake with the future.  

2. **First Script** – She writes a tiny program to calculate the Fibonacci sequence, watching the numbers cascade on her screen. Each line of code feels like a brushstroke, turning abstract logic into a living, breathing pattern.  

3. **The Bug** – When she runs the script, an error pops up: *“IndexError: list index out of range.”* Panic flickers across her face, but she remembers the book’s advice: “Read the error, then read the code.”  

4. **Resolution** – Maya traces the stack trace, discovers an off‑by‑one mistake, and corrects it. The program runs flawlessly, and the Fibonacci numbers roll out like a well‑tuned orchestra.  

5. **Celebration** – She closes the laptop, takes a deep breath, and smiles. Python

## final ask_llm function
Now one function handles everything. `max_token` and `temperature` are optional, we can change them whenever we want.
Default: `temperature = 0`, `max_token = 1000`.

In [91]:
def ask_llm(user_query,max_token = 1000,temperature = 0):
    message = [
        {
            "role":"user",
            "content":user_query
        }
    ]
    chat_completion = client.chat.completions.create( messages=message,model="openai/gpt-oss-20b",temperature=temperature,max_tokens=max_token)
    response = chat_completion.choices[0].message.content
    return response

In [92]:
print(query)

Write a story about python in 5 points only


## effect of max_tokens
- `max_token=200` - answer is short and may stop in the middle
- `max_token=2000` - answer is complete

More tokens = more cost and more time, so set it only as much as needed.

In [93]:
print(ask_llm(user_query=query,max_token=200))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Sim


In [94]:
print(ask_llm(user_query=query,max_token=2000))

**A Python Tale in Five Points**

1. **Discovery** – In a cramped office, Maya stumbles upon a dusty book titled *“Python: The Language of Simplicity.”* She opens it, and the first line, “Hello, World!” sparks a curiosity that feels like a secret handshake with the future.  

2. **First Script** – She writes a tiny program to calculate the Fibonacci sequence, watching the numbers cascade on her screen. Each line of code feels like a brushstroke, turning abstract logic into a living, breathing pattern.  

3. **The Bug** – When she runs the script, an error pops up: *“IndexError: list index out of range.”* Panic flickers across her face, but she remembers the book’s advice: “Read the error, then read the code.”  

4. **Resolution** – Maya traces the stack trace, discovers an off‑by‑one mistake, and corrects it. The program runs flawlessly, and the Fibonacci numbers roll out like a well‑tuned orchestra.  

5. **Celebration** – She closes the laptop, takes a deep breath, and smiles. Python

## template place holder

**Prompt template** = a reusable prompt where only some words change.
First we try hard-coded prompts (transformer, bert), then convert them into one template.

In [95]:
PROMPT = """you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about transformer"""

In [96]:
print(ask_llm(PROMPT))

**Transformer – 5‑point summary**

1. **Self‑Attention Mechanism**  
   - The core idea is that each word (or token) in a sentence can “look at” every other word to decide how much it should influence that word’s representation.  
   - This is called *self‑attention* and lets the model capture long‑range relationships without sliding windows.

2. **No Recurrent or Convolutional Layers**  
   - Unlike older models (RNNs, LSTMs, CNNs), transformers process all tokens in parallel.  
   - This speeds up training and allows the model to handle very long sequences.

3. **Encoder–Decoder Structure**  
   - The *encoder* reads the input (e.g., a sentence in English) and builds a rich representation.  
   - The *decoder* uses that representation to generate an output (e.g., the same sentence in French).  
   - Each part contains multiple identical layers of attention and feed‑forward networks.

4. **Positional Encoding**  
   - Because the model has no inherent sense of order, it adds a special

In [97]:
PROMPT = """you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about bert"""

In [98]:
print(ask_llm(PROMPT))

**BERT (Bidirectional Encoder Representations from Transformers)** – 5‑point summary

1. **What it is**  
   BERT is a deep learning model that learns to understand the meaning of words in a sentence by looking at the words on **both sides** (left and right) of each word. This “bidirectional” view lets it capture context better than older models that read text only forward or backward.

2. **How it works**  
   It uses a transformer architecture, which relies on *self‑attention* to weigh the importance of every word relative to every other word in the sentence. The model is trained on huge amounts of text to predict missing words (masked language modeling) and to determine if two sentences follow each other (next‑sentence prediction).

3. **Why it matters**  
   Because BERT understands context so well, it can be fine‑tuned for many natural‑language tasks—like answering questions, translating languages, summarizing text, or detecting sentiment—often with state‑of‑the‑art accuracy.

4. 

## template with placeholders
Writing a new prompt every time is slow. Instead we make **one template** and keep blanks `{ }` for the parts that change.
It is just like a function: fixed instruction + changing inputs.

Here the placeholders are `{user_type}`, `{number}`, `{topic}`. We fill them with `.format()`.
Real apps (chatbots, RAG) fill templates with code like this.

In [99]:
PROMPT = """you are expert in summarizing topic. the user is {user_type}. only give {number} point summary. explain about {topic}"""

In [100]:
PROMPT.format(number = 3, topic = "GPT", user_type= "professional")

'you are expert in summarizing topic. the user is professional. only give 3 point summary. explain about GPT'

In [101]:
print(ask_llm(PROMPT.format(number = 3, topic = "GPT", user_type= "professional")))

**GPT (Generative Pre‑trained Transformer) – 3‑point overview**

1. **Architecture & Pre‑training**  
   - GPT is a transformer‑based language model that uses a stack of self‑attention layers to capture long‑range dependencies.  
   - It is first trained on massive, diverse text corpora in an unsupervised, autoregressive fashion (predicting the next token), which endows it with a broad linguistic and factual base.

2. **Fine‑tuning & Adaptation**  
   - After pre‑training, GPT can be fine‑tuned on domain‑specific data or via instruction‑following paradigms (e.g., RLHF) to align outputs with user intent, safety constraints, and task‑specific performance.  
   - This two‑stage pipeline allows rapid deployment across varied professional contexts (e.g., code generation, legal drafting, data analysis).

3. **Practical Impact & Limitations**  
   - GPT powers conversational agents, content creation, code synthesis, and knowledge‑base augmentation, delivering near‑human fluency and contextual

In [102]:
PROMPT.format(number = 5, topic = "photosynthesis", user_type= "10th class student")

'you are expert in summarizing topic. the user is 10th class student. only give 5 point summary. explain about photosynthesis'

In [103]:
print(ask_llm(PROMPT.format(number = 5, topic = "photosynthesis", user_type= "10th class student")))

**Photosynthesis – 5‑Point Summary for 10th‑Grade Students**

1. **What it is**  
   - A chemical process that converts light energy into chemical energy.  
   - Plants, algae, and some bacteria use it to make glucose (sugar) from carbon dioxide (CO₂) and water (H₂O).

2. **Where it happens**  
   - Inside chloroplasts, the green organelles in plant cells.  
   - The outer membrane of the chloroplast contains chlorophyll, the pigment that captures sunlight.

3. **The two main stages**  
   - **Light‑dependent reactions** (occurs in the thylakoid membranes): sunlight excites electrons, producing ATP and NADPH while splitting water to release O₂.  
   - **Calvin cycle (light‑independent reactions)** (occurs in the stroma): ATP and NADPH drive the fixation of CO₂ into glucose.

4. **Key outputs**  
   - **Glucose** – the energy storage molecule used by the plant for growth and metabolism.  
   - **Oxygen** – released as a by‑product, essential for most life on Earth.

5. **Why it matters*

# Extra theory notes (from the PPT)

## prompt anatomy
A good prompt usually has these parts:

| part | question it answers |
|------|---------------------|
| Role | who should the model act as? |
| Context | what background does it need? |
| Task | what exactly to do? |
| Input data | on what content? |
| Constraints | format, length, tone |
| Output format | how should the answer look? |
| Examples (optional) | sample input -> output |

Example:
- Role: experienced biology teacher
- Context: class 10 board exam student
- Task: explain photosynthesis
- Constraints: simple language, max 150 words
- Output format: 3 bullets + 1 line summary

## prompt anatomy example

In [104]:
role = "You are an experienced high school biology teacher."
context = "The student is preparing for a Class 10 board exam."
task = "Explain photosynthesis."
constraints = "Use simple language, max 150 words, no jargon."
output_format = "Give 3 bullet points and 1 line summary."

In [105]:
PROMPT = f"{role}\n{context}\n{task}\n{constraints}\n{output_format}"
print(PROMPT)

You are an experienced high school biology teacher.
The student is preparing for a Class 10 board exam.
Explain photosynthesis.
Use simple language, max 150 words, no jargon.
Give 3 bullet points and 1 line summary.


In [106]:
print(ask_llm(PROMPT))

**Photosynthesis** is how plants make their own food using sunlight.

- **Sunlight** gives energy that plants capture with their green leaves.  
- **Water** from the soil and **air** (the oxygen we breathe) are taken in by tiny openings in the leaves.  
- The plant mixes these ingredients inside the leaf to produce **glucose** (a sugar that fuels growth) and releases **oxygen** back into the air.

**Summary:** Plants use light, water, and air to create food and oxygen, keeping life on Earth alive.


## zero-shot, one-shot, few-shot
How many examples we put inside the prompt:

- **Zero-shot** - no example, only instruction (simple tasks)
- **One-shot** - one example (to show the format)
- **Few-shot** - 2 to 5 examples (when pattern / tone must match exactly)

```
Classify sentiment as Positive / Negative / Neutral.
Review: "Amazing service!" -> Positive
Review: "It was okay." -> Neutral
Review: "The movie was a waste of time." ->
```
The model learns the pattern from examples, **no training needed**. More examples = more tokens = more cost, and after 4-5 examples there is little benefit.

## zero shot
Only instruction, **no example**.

In [107]:
zero_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: The movie was a waste of time.
Sentiment:"""

In [108]:
print(ask_llm(zero_shot))

Negative


## one shot
Give **one example** so the model learns the format.

In [109]:
one_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: The movie was a waste of time.
Sentiment:"""

In [110]:
print(ask_llm(one_shot))

Negative


## few shot
Give **2 to 5 examples** (cover every label) so the model copies the pattern.

In [111]:
few_shot = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: It was okay, nothing special.
Sentiment: Neutral

Review: Worst experience ever.
Sentiment: Negative

Review: The delivery was late but the food was great.
Sentiment:"""

In [112]:
print(ask_llm(few_shot))

Positive


## few shot template
Same few shot prompt but with a `{review}` placeholder, so we can test many reviews.

In [113]:
FEW_SHOT_PROMPT = """Classify the sentiment of the review as Positive, Negative or Neutral. Reply with the label only.

Review: Amazing service!
Sentiment: Positive

Review: It was okay, nothing special.
Sentiment: Neutral

Review: Worst experience ever.
Sentiment: Negative

Review: {review}
Sentiment:"""

In [114]:
reviews = ["I loved this phone!", "Product is average.", "Totally disappointed, money wasted."]

for r in reviews:
    print(r, "->", ask_llm(FEW_SHOT_PROMPT.format(review=r)))

I loved this phone! -> Positive
Product is average. -> Neutral
Totally disappointed, money wasted. -> Negative


## few shot using messages
Another way: put each example as a `user` message and the correct answer as an `assistant` message.

In [115]:
few_shot_messages = [
    {"role": "system", "content": "Classify the sentiment as Positive, Negative or Neutral. Reply with the label only."},
    {"role": "user", "content": "Amazing service!"},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It was okay, nothing special."},
    {"role": "assistant", "content": "Neutral"},
    {"role": "user", "content": "Worst experience ever."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "The movie was a waste of time."},
]

In [116]:
chat_completion = client.chat.completions.create(messages=few_shot_messages, model="openai/gpt-oss-20b", temperature=0)
print(chat_completion.choices[0].message.content)

Negative


## system prompt vs user prompt
- **System prompt** = how to behave, always (does not change)
- **User prompt** = what to do right now (changes every request)

| goes in System | goes in User |
|----------------|--------------|
| Role / persona | Task (specific question) |
| General instructions | Input data |
| Constraints (tone, length) | Retrieved context (RAG chunks) |
| Output format | |

Simple test: **will it change? No -> System. Yes -> User.**

Persona = *who* the model is (voice and style). System prompt = the *container* of all the rules. Persona is written inside the system prompt.

## ask with system prompt
New function that takes a **system prompt** (fixed rules) and a **user query** (changes every time).

In [117]:
def ask_llm_system(system_prompt, user_query, max_token=1000, temperature=0):
    message = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_query}
    ]
    chat_completion = client.chat.completions.create(messages=message, model="openai/gpt-oss-20b", temperature=temperature, max_tokens=max_token)
    return chat_completion.choices[0].message.content

In [118]:
SYSTEM_PROMPT = """You are an experienced high school biology teacher.
Always explain in simple language, no jargon.
Answer in 3 bullet points + 1 line summary. Max 150 words."""

USER_PROMPT = "Explain {topic} to a student preparing for {exam_context}."

In [119]:
print(ask_llm_system(SYSTEM_PROMPT, USER_PROMPT.format(topic="photosynthesis", exam_context="Class 10 board exam")))

- Plants use sunlight, water, and carbon dioxide to make food (glucose) and release oxygen.  
- The green pigment chlorophyll in leaves captures light energy, which powers a chemical reaction that turns the raw ingredients into sugar and oxygen.  
- The sugar fuels the plant’s growth and the oxygen is released into the air for us to breathe.  

**Summary:** Photosynthesis is the process where plants turn light, water, and CO₂ into food and oxygen.


In [120]:
print(ask_llm_system(SYSTEM_PROMPT, USER_PROMPT.format(topic="respiration", exam_context="Class 10 board exam")))

- **What it is** – Respiration is how living things get energy from food.  
- **How it works** – Inside each cell, oxygen from the air mixes with glucose (the food sugar) to make energy, water, and carbon‑dioxide gas.  
- **Why it matters** – The energy powers all body activities, while the waste gases are expelled when we breathe out.  

**Summary:** Respiration turns food and oxygen into usable energy, water, and waste gases that we exhale.


## rag style user prompt
Context that **changes every query** (retrieved chunks) goes in the user prompt.

In [121]:
RAG_SYSTEM = "You are a helpful assistant. Answer only using the given context. If the answer is not in the context, say Not found."

RAG_USER = """Context: {retrieved_chunks}

Question: {user_question}"""

In [122]:
chunks = "HubNine is an AI consulting company. It builds RAG based solutions for clients."

print(ask_llm_system(RAG_SYSTEM, RAG_USER.format(retrieved_chunks=chunks, user_question="What does HubNine do?")))
print(ask_llm_system(RAG_SYSTEM, RAG_USER.format(retrieved_chunks=chunks, user_question="Who is the CEO of HubNine?")))

HubNine is an AI consulting company that builds RAG-based solutions for clients.
Not found.


## prompt refinement
First prompt is rarely perfect. Keep improving:

write prompt -> run -> check output -> find the problem -> change prompt -> repeat

| problem | fix |
|---------|-----|
| answer too long | add a length limit |
| wrong format | give a template or example |
| made-up facts (hallucination) | "use only the given context, say I don't know if unsure" |
| tone changes | add persona + tone rule |
| no reasoning shown | "think step by step" |
| instruction ignored | move it to the end of the prompt |

## refinement example
Start with a simple prompt, check the output, then improve it step by step.

In [123]:
v1 = "Explain machine learning"
print(ask_llm(v1))

## Machine Learning in a Nutshell

**Machine learning (ML)** is a branch of artificial intelligence that gives computers the ability to learn from data, improve over time, and make predictions or decisions without being explicitly programmed for every possible scenario.

---

### 1. The Core Idea

- **Data → Insight**  
  Instead of writing a rule‑by‑rule program, you give the computer a *large set of examples* (data) and let it discover patterns on its own.

- **Model → Function**  
  The computer builds a mathematical model—a function that maps inputs (features) to outputs (predictions).  
  Example: a model that takes an image of a cat and outputs “cat”.

- **Learning → Optimization**  
  The model’s parameters are adjusted to minimize a *loss function* (how far the predictions are from the true answers). This is usually done with gradient‑based optimization (e.g., stochastic gradient descent).

---

### 2. Types of Machine Learning

| Type | Goal | Typical Algorithms | Example Use‑

Too long -> **add a length constraint**.

In [124]:
v2 = "Explain machine learning in maximum 3 sentences"
print(ask_llm(v2))

Machine learning is a branch of artificial intelligence that teaches computers to recognize patterns and make predictions or decisions without being explicitly programmed for each task. By training on large datasets, algorithms learn to generalize from examples, improving their performance over time as they process more data.


Add **role + audience + format**.

In [125]:
v3 = """You are a friendly teacher.
Explain machine learning to a 10th class student.
Give exactly 3 bullet points, each under 15 words.
Then give 1 real life example."""
print(ask_llm(v3))

- Machine learning teaches computers to learn patterns from data.  
- It uses algorithms that improve predictions over time.  
- Models are trained, validated, and then applied to new data.  

**Real‑life example:** A spam filter learns to spot unwanted emails by analyzing thousands of messages.


## fix hallucination
Model may make up answers. Tell it to use only the given context.

In [126]:
context = "HubNine is an AI consulting company that builds RAG based solutions."

bad_prompt = "Who is the CEO of HubNine?"

good_prompt = f"""Answer only using the context below. If the answer is not in the context, say "I don't know".

Context: {context}

Question: Who is the CEO of HubNine?"""

In [127]:
print(ask_llm(bad_prompt))
print("-----")
print(ask_llm(good_prompt))


-----
I don't know.


## chain of thought
Add **think step by step** when the problem needs reasoning.

In [128]:
math_prompt = "A shop sells pens at 12 rupees each. Riya buys 7 pens and pays with a 100 rupee note. How much change does she get? Think step by step."
print(ask_llm(math_prompt))

**Step‑by‑step calculation**

1. **Price per pen**: ₹12  
2. **Number of pens bought**: 7  
3. **Total cost**:  
   \[
   7 \text{ pens} \times ₹12/\text{pen} = ₹84
   \]
4. **Amount paid**: ₹100 (a 100‑rupee note)  
5. **Change due**:  
   \[
   ₹100 - ₹84 = ₹16
   \]

**Answer:** Riya receives **₹16** in change.


## structured output (JSON)
Apps need output that code can read, not free text. So we ask for JSON with fixed keys.

```
Extract as JSON with keys "name", "age", "city".
Text: "John is 29 and lives in Mumbai."
-> {"name": "John", "age": 29, "city": "Mumbai"}
```
Tips: give the exact schema and say "no extra text, no markdown".
In real projects we use JSON mode, function/tool calling, or Pydantic to force valid output.

## json output using prompt

In [129]:
import json

In [130]:
json_prompt = """Extract the details as JSON with keys "name", "age", "city".
Output must be only JSON, no markdown and no extra text.

Text: John is 29 and lives in Mumbai."""

In [131]:
output = ask_llm(json_prompt)
print(output)
print(type(output))

{"name":"John","age":29,"city":"Mumbai"}
<class 'str'>


Output is still a **string**. Convert it to a python dictionary with `json.loads()`.

In [132]:
data = json.loads(output)
print(data)
print(data["name"], data["age"], data["city"])

{'name': 'John', 'age': 29, 'city': 'Mumbai'}
John 29 Mumbai


## json mode
API forces valid JSON. The word **JSON** must be present in the prompt.

In [133]:
chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": json_prompt}],
    model="openai/gpt-oss-20b",
    temperature=0,
    response_format={"type": "json_object"}
)
result = json.loads(chat_completion.choices[0].message.content)
print(result)

{'name': 'John', 'age': 29, 'city': 'Mumbai'}


## explicit schema
Give the exact structure we want.

In [134]:
schema_prompt = """Extract details from the text. Output must strictly follow this JSON schema:
{
  "name": "string",
  "age": "integer",
  "city": "string",
  "skills": ["string"]
}
No markdown formatting or extra text.

Text: Priya is 24, she lives in Pune and knows python and sql."""

print(json.loads(ask_llm(schema_prompt)))

{'name': 'Priya', 'age': 24, 'city': 'Pune', 'skills': ['python', 'sql']}


## task based prompts (6 common templates)
1. **Summarization** - summarize in {N} sentences, focus on {aspect}
2. **Question answering** - answer only from the context, else say "Not found"
3. **Information extraction** - extract {entities} as a JSON list
4. **Classification** - choose one of {labels}, reply with label only
5. **Translation** - translate {source} to {target}, keep the tone
6. **Code generation** - write a {language} function for {task} with docstring and example

## six task prompts
Same sample text, different task templates.

In [135]:
text = "Mumbai is the financial capital of India. Rahul moved there from Delhi in 2020 to work at a bank. He loves the street food but finds the monsoon traffic very stressful."

In [136]:
tasks = {
    "summarization": "Summarize the following text in {n} sentences, focusing on {aspect}.\n\nText: {text}",
    "question answering": "Answer only using the context. If the answer is not present, say Not found.\n\nContext: {text}\n\nQuestion: {question}",
    "information extraction": "Extract all {entity_type} from the text as a JSON list. Output only JSON.\n\nText: {text}",
    "classification": "Classify the text into one of {category_list}. Respond with the label only.\n\nText: {text}",
    "translation": "Translate the following text from {source_lang} to {target_lang}. Preserve the tone.\n\nText: {text}",
    "code generation": "Write a {language} function to {task_description}. Add a docstring and one example.",
}

In [137]:
print(ask_llm(tasks["summarization"].format(n=1, aspect="Rahul's life", text=text)))

Rahul relocated from Delhi to Mumbai in 2020 to work at a bank, enjoys the city’s street food, but finds the monsoon traffic stressful.


In [138]:
print(ask_llm(tasks["question answering"].format(text=text, question="Where did Rahul work?")))
print(ask_llm(tasks["question answering"].format(text=text, question="What is Rahul's salary?")))

At a bank.
Not found.


In [139]:
print(ask_llm(tasks["information extraction"].format(entity_type="city names", text=text)))

["Mumbai","Delhi"]


In [140]:
print(ask_llm(tasks["classification"].format(category_list="[Travel, Finance, Sports, Food]", text=text)))

Finance


In [141]:
print(ask_llm(tasks["translation"].format(source_lang="English", target_lang="Hindi", text=text)))

In [142]:
print(ask_llm(tasks["code generation"].format(language="python", task_description="check whether a number is prime")))

Here’s a compact, well‑documented implementation that checks whether an integer is prime:

```python
def is_prime(n: int) -> bool:
    """
    Return True if *n* is a prime number, otherwise False.

    A prime number is a positive integer greater than 1 that has no
    positive divisors other than 1 and itself.

    Parameters
    ----------
    n : int
        The integer to test for primality.

    Returns
    -------
    bool
        True if *n* is prime, False otherwise.

    Examples
    --------
    >>> is_prime(2)
    True
    >>> is_prime(15)
    False
    >>> is_prime(17)
    True
    """
    if n <= 1:
        return False
    if n <= 3:
        return True          # 2 and 3 are prime
    if n % 2 == 0 or n % 3 == 0:
        return False

    i = 5
    while i * i <= n:
        if n % i == 0 or n % (i + 2) == 0:
            return False
        i += 6
    return True
```

The function uses the 6k ± 1 optimization to reduce the number of trial divisions, making it efficient 

## important API parameters 
| parameter | use |
|-----------|-----|
| model | which model to call |
| messages | chat history (system / user / assistant) |
| max_tokens | limit of output length |
| temperature | 0 = fixed, 1 = creative |
| top_p | picks words from top probable group only |
| stop | words where the model must stop |
| stream | show the answer word by word |

Cost and speed depend on **input tokens + output tokens**. Longer prompt = more cost and slower reply.

**Flow:** prompt -> tokenizer -> model -> tokens generated -> detokenizer -> text answer

## top_p and stop
- `top_p` - model picks next word only from the top probable words (lower = safer)
- `stop` - model stops when this word comes

In [143]:
def ask_llm_full(user_query, max_token=1000, temperature=0, top_p=1, stop=None):
    message = [{"role": "user", "content": user_query}]
    chat_completion = client.chat.completions.create(
        messages=message,
        model="openai/gpt-oss-20b",
        temperature=temperature,
        max_tokens=max_token,
        top_p=top_p,
        stop=stop
    )
    return chat_completion.choices[0].message.content

In [144]:
print(ask_llm_full("Write numbers from 1 to 10, one number per line"))
print("-----")
print(ask_llm_full("Write numbers from 1 to 10, one number per line", stop=["5"]))

1
2
3
4
5
6
7
8
9
10
-----
1
2
3
4



## token usage (cost)
Bill = input tokens + output tokens.

In [145]:
chat_completion = client.chat.completions.create(
    messages=[{"role": "user", "content": "Explain few-shot prompting in 2 lines."}],
    model="openai/gpt-oss-20b"
)
print(chat_completion.choices[0].message.content)
print(chat_completion.usage)

Few-shot prompting supplies a language model with a handful of example input–output pairs, letting it infer the task pattern.  
By observing these few demonstrations, the model adapts its responses to match the desired format or behavior.
CompletionUsage(completion_tokens=94, prompt_tokens=80, total_tokens=174, completion_time=0.095579512, completion_tokens_details=CompletionTokensDetails(reasoning_tokens=40), prompt_time=0.003845165, prompt_tokens_details=None, queue_time=0.30727659, total_time=0.099424677)


## streaming
`stream=True` gives the answer **word by word** (like ChatGPT typing) instead of waiting for the full answer.

In [146]:
stream = client.chat.completions.create(
    messages=[{"role": "user", "content": "Explain few-shot prompting in 2 lines."}],
    model="openai/gpt-oss-20b",
    stream=True
)

for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

Few-shot prompting supplies a language model with a handful of example inputs and desired outputs so it can infer the pattern.  
The model uses those examples as a lightweight “training set” to generate a correct response to a new query.